# trajectory progression synthetic dataset benchmark

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Score signed progression correlations (Figure 4E–F)

In [ ]:
# Zoetrope | Synthetic developmental crescent | Results figure: progression panel
#
# Two vertically combined plots, 3.35 x 5.50 inches:
#   1. Pooled Spearman correlation with simulator molecular progression
#   2. Per-tissue Spearman correlation, displayed as Tissue 2 through Tissue 10
#
# IMPORTANT: STORIES' native differentiation potential J is used as NEGATIVE J
# in both comparisons. The sign is fixed here, independently of the truth.
# Signed Spearman, NOT absolute Spearman, makes this orientation distinguishable.
# No spatial tissue percentile heatmaps in this cell (reserved for next panel).
# This cell does NOT refit methods, integrate trajectories, or alter frozen files.
# It uses the evaluation-only per-cell output of the original benchmark; an
# exact aggregate-file fallback is provided if the per-cell output is missing.
#
# Exports 3.35 x 5.50 inch SVG/PDF/PNG, signed and absolute score tables,
# and an audit manifest. PowerPoint-safe SVG is produced using text paths.

from pathlib import Path
import hashlib
import json
import os
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FormatStrFormatter
from scipy.stats import spearmanr

# =====================================================================
# 1. PATHS AND CONSTANTS
# =====================================================================
PROJECT = Path(os.environ.get(
    'STP_PROJECT', os.environ["STP_PROJECT"]
)).expanduser()
DATA = Path(os.environ.get(
    'CRESCENT_DATA_ROOT',
    str(PROJECT / 'Experiments/simulation validation/Synthetic benchmarking/'
                  'experiments/synthetic_developmental_crescent_v1')
)).expanduser()

SOURCE_DEFAULT = (
    PROJECT / 'experiments/synthetic_benchmark/publication_results/'
              'progression_TI_vs_STORIES_seed20261023_native_potential_v3'
)
SOURCE = Path(os.environ.get('PROGRESSION_BENCHMARK_OUT', str(SOURCE_DEFAULT))).expanduser()
OUT = Path(os.environ.get(
    'SYNTHETIC_PROGRESSION_PANEL_OUT',
    str(PROJECT / 'experiments/synthetic_benchmark/publication_results/'
                  'results_figure_panel_C_progression_negative_STORIES')
)).expanduser()

FIG_W = 3.35
FIG_H = 5.50
DPI = 450
STEM = 'FIG_synthetic_progression_pooled_per_tissue_STORIES_negative_potential'

# Sign convention is predeclared, not selected by looking at the outcome.
STORIES_POTENTIAL_SIGN = -1.0
assert STORIES_POTENTIAL_SIGN == -1.0

RED = '#D62728'       # Zoetrope
BLUE = '#4C78A8'      # STORIES (-J)
INK = '#23313B'
GRID = '#D2D7DC'
ORDER = ['Zoetrope', 'STORIES (-J)']
COLORS = [RED, BLUE]


def require(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Required frozen evaluation file not found: {path}')
    return path


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


manifest_path = require(DATA / 'manifests/dataset_manifest.json')
manifest = json.loads(manifest_path.read_text())
assert manifest['dataset_id'] == 'synthetic_developmental_crescent_v1'
assert int(manifest['seed']) == 20261023
assert manifest['status'] == 'frozen'

# =====================================================================
# 2. LOAD EXISTING FROZEN EVALUATION; FLIP STORIES, NEVER FIT/SELECT SIGN
# =====================================================================
value_file = SOURCE / 'progression_values_evaluation_only.csv.gz'
original_pooled_path = SOURCE / 'pooled_absolute_spearman.csv'
original_tissue_path = SOURCE / 'per_tissue_absolute_spearman.csv'

input_hashes = {'dataset_manifest': sha256(manifest_path)}

if value_file.is_file():
    # This file was created by the uploaded original progression benchmark
    # *after* all prediction arrays were frozen. Reading it here is evaluation
    # and plotting only; it is not used to train or adapt the predictions.
    values = pd.read_csv(value_file)
    required = {
        'cell_id', 'tissue', 'rho_true', 'rho_spatial_ti',
        'stories_native_potential'
    }
    assert required.issubset(values.columns), required - set(values.columns)
    assert len(values) == 36425 and values.cell_id.astype(str).is_unique
    assert np.isfinite(values[['rho_true', 'rho_spatial_ti',
                               'stories_native_potential']].to_numpy(float)).all()
    tissue = values['tissue'].to_numpy(int)
    assert np.array_equal(np.unique(tissue), np.arange(10))
    truth = values['rho_true'].to_numpy(float)
    zoetrope = values['rho_spatial_ti'].to_numpy(float)
    stories_negative = STORIES_POTENTIAL_SIGN * values['stories_native_potential'].to_numpy(float)
    valid = tissue > 0    # at t=0, true progression is identically zero
    assert np.allclose(truth[tissue == 0], 0, atol=1e-8)

    def corr(a, b):
        result = float(spearmanr(a, b).statistic)
        assert np.isfinite(result), 'Undefined Spearman coefficient'
        return result

    pooled_rows = [
        {'method': 'Zoetrope', 'signed_spearman': corr(truth[valid], zoetrope[valid]),
         'n_cells': int(valid.sum())},
        {'method': 'STORIES (-J)', 'signed_spearman': corr(truth[valid], stories_negative[valid]),
         'n_cells': int(valid.sum())}
    ]
    tissue_rows = []
    for k in range(1, 10):
        take = tissue == k
        tissue_rows += [
            {'tissue_id': k, 'display_tissue': k + 1, 'method': 'Zoetrope',
             'signed_spearman': corr(truth[take], zoetrope[take]),
             'n_cells': int(take.sum())},
            {'tissue_id': k, 'display_tissue': k + 1, 'method': 'STORIES (-J)',
             'signed_spearman': corr(truth[take], stories_negative[take]),
             'n_cells': int(take.sum())},
        ]
    input_hashes['per_cell_evaluation'] = sha256(value_file)
    read_mode = 'per_cell_frozen_evaluation'

else:
    # The original cell also froze both signed and absolute Spearman scores.
    # Spearman(Y,-J)=-Spearman(Y,J) identically; no predictions or truth
    # need to be opened to invert this author-defined potential orientation.
    old_pool = pd.read_csv(require(original_pooled_path))
    old_tissue = pd.read_csv(require(original_tissue_path))
    assert {'method', 'signed_spearman', 'absolute_spearman', 'n_cells'} <= set(old_pool)
    assert {'method', 'signed_spearman', 'absolute_spearman',
            'tissue', 'n_cells'} <= set(old_tissue)
    assert set(old_pool.method) == {'Spatial TI', 'STORIES'}
    assert set(old_tissue.method) == {'Spatial TI', 'STORIES'}
    assert set(old_tissue.tissue.astype(int)) == set(range(1, 10))
    map_names = {'Spatial TI': 'Zoetrope', 'STORIES': 'STORIES (-J)'}
    pooled_rows = []
    for _, row in old_pool.iterrows():
        name = map_names[row['method']]
        corr_value = float(row['signed_spearman'])
        if row['method'] == 'STORIES':
            corr_value = -corr_value
        pooled_rows.append({'method': name, 'signed_spearman': corr_value,
                            'n_cells': int(row['n_cells'])})
    tissue_rows = []
    for _, row in old_tissue.iterrows():
        k = int(row['tissue'])
        name = map_names[row['method']]
        corr_value = float(row['signed_spearman'])
        if row['method'] == 'STORIES':
            corr_value = -corr_value
        tissue_rows.append({'tissue_id': k, 'display_tissue': k + 1,
                            'method': name, 'signed_spearman': corr_value,
                            'n_cells': int(row['n_cells'])})
    input_hashes['pooled_original'] = sha256(original_pooled_path)
    input_hashes['tissue_original'] = sha256(original_tissue_path)
    read_mode = 'original_signed_aggregate_scores'

pooled = pd.DataFrame(pooled_rows).set_index('method').loc[ORDER].reset_index()
by_tissue = (pd.DataFrame(tissue_rows)
             .sort_values(['tissue_id', 'method'])
             .reset_index(drop=True))
assert len(pooled) == 2 and len(by_tissue) == 18
for frame in (pooled, by_tissue):
    frame['absolute_spearman'] = np.abs(frame['signed_spearman'])
    assert np.isfinite(frame['signed_spearman']).all()
    assert frame['signed_spearman'].between(-1, 1).all()

# Confirm reorientation is precisely a sign flip and no original results
# have been changed. When originals exist, check original signed estimates.
if original_pooled_path.is_file() and original_tissue_path.is_file():
    original_pool = pd.read_csv(original_pooled_path).set_index('method')
    original_by_tissue = pd.read_csv(original_tissue_path).set_index(['tissue', 'method'])
    for name, old_name, sign in (
        ('Zoetrope', 'Spatial TI', +1),
        ('STORIES (-J)', 'STORIES', -1),
    ):
        observed = float(pooled.set_index('method').loc[name, 'signed_spearman'])
        expected = sign * float(original_pool.loc[old_name, 'signed_spearman'])
        np.testing.assert_allclose(observed, expected, atol=5e-8, rtol=0)
        for k in range(1, 10):
            observed = float(by_tissue.loc[
                (by_tissue.tissue_id == k) & (by_tissue.method == name),
                'signed_spearman'].iloc[0])
            expected = sign * float(original_by_tissue.loc[(k, old_name), 'signed_spearman'])
            np.testing.assert_allclose(observed, expected, atol=5e-8, rtol=0)
    input_hashes['pooled_original'] = sha256(original_pooled_path)
    input_hashes['tissue_original'] = sha256(original_tissue_path)

print('\nPOOLED PROGRESSION RECOVERY — SIGNED SPEARMAN')
print(pooled.to_string(index=False, float_format=lambda x: f'{x:.5f}'))
print('\nPER-TISSUE PROGRESSION RECOVERY — SIGNED SPEARMAN')
print(by_tissue[['display_tissue','method','signed_spearman']].to_string(
    index=False, float_format=lambda x: f'{x:.5f}'))

# =====================================================================
# 3. PUBLICATION STYLE — CONSISTENT WITH RESULTS PANELS A AND B
# =====================================================================
available_fonts = {font.name for font in fm.fontManager.ttflist}
font_candidates = ['Arial', 'Liberation Sans', 'Arimo', 'Nimbus Sans', 'DejaVu Sans']
FONT = next((f for f in font_candidates if f in available_fonts), 'DejaVu Sans')

mpl.rcParams.update({
    'font.family': FONT,
    'font.size': 10.5,
    'axes.labelsize': 11.6,
    'axes.titlesize': 12.0,
    'xtick.labelsize': 10.3,
    'ytick.labelsize': 10.3,
    'axes.linewidth': 0.75,
    'axes.edgecolor': INK,
    'axes.labelcolor': INK,
    'xtick.color': INK,
    'ytick.color': INK,
    'text.color': INK,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'path',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'savefig.bbox': None,
})

# Inch-based coordinates: exact 3.35 x 5.50 in page; no tight cropping.
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')
# Move both axes left by 0.10 in to use excess left margin and prevent
# the long bottom title extending beyond the right edge of the SVG.
# Keep all text sizes and the physical canvas (3.35 x 5.50 in) unchanged.
BAR_LEFT, BAR_BOTTOM, BAR_WIDTH, BAR_HEIGHT = 0.62, 3.49, 2.39, 1.40
LINE_LEFT, LINE_BOTTOM, LINE_WIDTH, LINE_HEIGHT = 0.62, 0.61, 2.39, 2.06

# Shared y-limit: show zero and any negative correlations unambiguously.
all_scores = np.r_[pooled.signed_spearman.to_numpy(float),
                   by_tissue.signed_spearman.to_numpy(float)]
if all_scores.min() < -0.05:
    YLIM = (-1.05, 1.05)
    YTICKS = [-1, -.5, 0, .5, 1]
else:
    YLIM = (-.065, 1.05)
    YTICKS = [0, .25, .5, .75, 1]

# Compact shared figure heading
fig.text(0.51, 5.34 / FIG_H, 'Trajectory progression',
         fontsize=12.2, fontweight='bold', ha='center', va='center')

# -------------------- TOP: POOLED CORRELATION --------------------
ax_bar = fig.add_axes([BAR_LEFT / FIG_W, BAR_BOTTOM / FIG_H,
                       BAR_WIDTH / FIG_W, BAR_HEIGHT / FIG_H])
bar_values = pooled.set_index('method').loc[ORDER, 'signed_spearman'].to_numpy(float)
bar_x = np.arange(2)
ax_bar.bar(bar_x, bar_values, width=.62, color=COLORS,
           edgecolor='none', zorder=3)
ax_bar.set_xlim(-.55, 1.55)
ax_bar.set_ylim(*YLIM)
ax_bar.set_yticks(YTICKS)
ax_bar.yaxis.set_major_formatter(FormatStrFormatter('%g'))
ax_bar.set_xticks(bar_x)
ax_bar.set_xticklabels(ORDER, fontsize=10.3, fontweight='bold')
ax_bar.set_ylabel('Spearman correlation', fontsize=11.3, labelpad=4)
ax_bar.set_title('Pooled progression correlation', fontsize=11.7, fontweight='bold', pad=8)
ax_bar.axhline(0, linewidth=.65, color=INK, alpha=.7, zorder=2)
ax_bar.grid(axis='y', color=GRID, alpha=.50, linewidth=.55)
ax_bar.set_axisbelow(True)
ax_bar.spines[['top', 'right']].set_visible(False)
ax_bar.tick_params(axis='x', length=0, pad=4)
ax_bar.tick_params(axis='y', length=3, width=.7, pad=2)
for xi, val in zip(bar_x, bar_values):
    # Write values just outside bars, both for positive and negative values.
    ax_bar.annotate(f'{val:+.3f}', (xi, val),
                    xytext=(0, 4 if val >= 0 else -5),
                    textcoords='offset points', ha='center',
                    va='bottom' if val >= 0 else 'top',
                    fontsize=9.4, color=INK)

# -------------------- BOTTOM: BY TISSUE --------------------
ax_line = fig.add_axes([LINE_LEFT / FIG_W, LINE_BOTTOM / FIG_H,
                        LINE_WIDTH / FIG_W, LINE_HEIGHT / FIG_H])
for name, color, marker in zip(ORDER, COLORS, ['o', 's']):
    sub = by_tissue.loc[by_tissue.method.eq(name)].sort_values('tissue_id')
    ax_line.plot(sub.display_tissue, sub.signed_spearman,
                 color=color, lw=2.0, marker=marker, markersize=5.6,
                 markeredgecolor='white', markeredgewidth=.65,
                 label=name, zorder=3)
ax_line.set_xlim(1.65, 10.35)
ax_line.set_xticks(np.arange(2, 11))
ax_line.set_ylim(*YLIM)
ax_line.set_yticks(YTICKS)
ax_line.yaxis.set_major_formatter(FormatStrFormatter('%g'))
ax_line.set_xlabel('Tissue', fontsize=11.6, labelpad=5)
ax_line.set_ylabel('Spearman correlation', fontsize=11.3, labelpad=4)
ax_line.set_title('Progression correlation by tissue', fontsize=11.7, fontweight='bold', pad=8)
ax_line.axhline(0, linewidth=.65, color=INK, alpha=.7, zorder=2)
ax_line.grid(axis='y', color=GRID, alpha=.50, linewidth=.55)
ax_line.set_axisbelow(True)
ax_line.spines[['top', 'right']].set_visible(False)
ax_line.tick_params(axis='x', length=3, width=.7, pad=3)
ax_line.tick_params(axis='y', length=3, width=.7, pad=2)
# The pooled bars establish the same two color identities; omitting a second
# legend keeps the small line plot fully visible at journal print size.

# =====================================================================
# 4. HARD CANVAS BOUNDARY CHECK AND EXPORT
# =====================================================================
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
bounds = fig.get_tightbbox(renderer).extents
print('\nCanvas:', FIG_W, 'x', FIG_H, 'in')
print('Rendered content bounds:', [round(float(v), 3) for v in bounds])
tol = 0.008
assert bounds[0] >= -tol and bounds[1] >= -tol, (
    'Plot elements extend beyond left/bottom edge', bounds)
assert bounds[2] <= FIG_W + tol and bounds[3] <= FIG_H + tol, (
    'Plot elements extend beyond right/top edge', bounds)

OUT.mkdir(parents=True, exist_ok=True)
for ext in ('svg', 'pdf', 'png'):
    path = OUT / f'{STEM}.{ext}'
    fig.savefig(path, dpi=DPI, bbox_inches=None, facecolor='white')
    print('SAVED:', path)

svg_path = OUT / f'{STEM}.svg'
root = ET.parse(svg_path).getroot()
print('SVG dimensions:', root.attrib.get('width'), root.attrib.get('height'))

pooled.to_csv(OUT / f'{STEM}_pooled.csv', index=False)
by_tissue.to_csv(OUT / f'{STEM}_by_tissue.csv', index=False)
receipt = {
    'dataset_id': 'synthetic_developmental_crescent_v1',
    'dataset_seed': 20261023,
    'source_directory': str(SOURCE),
    'evaluation_source_mode': read_mode,
    'primary_score': 'SIGNED Spearman correlation with true molecular progression',
    'sign_convention': 'STORIES negative native potential -J (fixed independently of ground truth)',
    'spatial_ti_quantity': 'Fisher molecular arclength progression rho',
    'stories_quantity': 'Negative native expression-state potential -J, NOT Fisher path length',
    'scores_are_comparable_as_ranking_associations_only': True,
    'tissue_ids_evaluated': list(range(1, 10)),
    'tissue_labels_displayed': list(range(2, 11)),
    'tissue_id_zero_excluded': 'ground-truth cumulative progression = 0 at starting time',
    'figure_inches': [FIG_W, FIG_H],
    'input_sha256': input_hashes,
    'font': FONT,
}
(OUT / f'{STEM}_receipt.json').write_text(
    json.dumps(receipt, sort_keys=True, indent=2) + '\n')
print('\nCompleted:', OUT)
plt.show()
plt.close(fig)

## Render tissue progression percentiles (Figure 4G)

In [ ]:
# Zoetrope | Synthetic developmental crescent | Progression tissue maps
# Three vertically stacked maps: ground truth, Zoetrope, STORIES (-J).
# Publication panel: exactly 3.35 x 5.50 inches.
# READ-ONLY evaluation/visualization: no training, no oracle used to fit models.
# Uses existing frozen progression outputs; tissue chosen by preexisting protocol.

from pathlib import Path
import os
import json
import hashlib
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import rankdata

# ============================================================
# 1. FROZEN INPUTS AND OUTPUT DIRECTORY
# ============================================================

PROJECT = Path(os.getenv(
    'STP_PROJECT',
    os.environ["STP_PROJECT"]
)).expanduser()
DATA = Path(os.getenv(
    'CRESCENT_DATA_ROOT',
    str(PROJECT / 'Experiments/simulation validation/'
                  'Synthetic benchmarking/experiments/'
                  'synthetic_developmental_crescent_v1')
)).expanduser()
IN_DIR = Path(os.getenv(
    'PROGRESSION_BENCHMARK_IN',
    str(PROJECT / 'experiments/synthetic_benchmark/publication_results/'
                  'progression_TI_vs_STORIES_seed20261023_native_potential_v3')
)).expanduser()
OUT = Path(os.getenv(
    'PROGRESSION_VERTICAL_PANEL_OUT',
    str(PROJECT / 'experiments/synthetic_benchmark/publication_results/'
                  'progression_TI_vs_STORIES_negative_potential_vertical_panel')
)).expanduser()

FIG_W, FIG_H = 3.35, 5.50
DPI = 450

# Publication typography: 12 pt axis label / 10.5 pt colorbar numbers.
PANEL_TITLE_SIZE = 11.5
MAP_TITLE_SIZE = 11.1
COLORBAR_TICK_SIZE = 10.5
COLORBAR_LABEL_SIZE = 11.0
INK = '#23313B'


def require(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing frozen input: {path}')
    return path


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def percentile_rank(x):
    x = np.asarray(x, dtype=float)
    assert len(x) > 0 and np.isfinite(x).all()
    return (rankdata(x, method='average') - 0.5) / len(x)


values_path = require(IN_DIR / 'progression_values_evaluation_only.csv.gz')
provenance_path = require(IN_DIR / 'provenance.json')
observed_path = require(DATA / 'train_only/public/observed_cells.csv.gz')
manifest_path = require(DATA / 'manifests/dataset_manifest.json')

manifest = json.loads(manifest_path.read_text())
assert manifest['dataset_id'] == 'synthetic_developmental_crescent_v1'
assert int(manifest['seed']) == 20261023
assert manifest['status'] == 'frozen'
assert sha256(observed_path) == manifest['file_sha256'][
    'train_only/public/observed_cells.csv.gz'
], 'Frozen observed-data hash mismatch'

provenance = json.loads(provenance_path.read_text())
assert provenance['dataset_id'] == manifest['dataset_id']
assert int(provenance['seed']) == 20261023
if 'source_sha256' in provenance and 'observed' in provenance['source_sha256']:
    assert provenance['source_sha256']['observed'] == sha256(observed_path), (
        'Progression benchmark and observed geometry are from different frozen inputs'
    )

# ============================================================
# 2. ALIGN CELL VALUES WITH OBSERVED TISSUE GEOMETRY
# ============================================================

vals = pd.read_csv(values_path)
obs = pd.read_csv(observed_path, usecols=['cell_id', 'tissue_id', 'x', 'y'])
vals['cell_id'] = vals['cell_id'].astype(str)
obs['cell_id'] = obs['cell_id'].astype(str)

required = {'cell_id', 'tissue', 'rho_true', 'rho_spatial_ti',
            'stories_native_potential'}
assert required.issubset(vals.columns), required - set(vals.columns)
assert vals['cell_id'].is_unique and obs['cell_id'].is_unique

# IMPORTANT: obs uses 'tissue_id', whereas vals uses 'tissue'.
# Use an explicit left/right key to avoid KeyError: 'tissue'.
df = obs.merge(
    vals,
    left_on=['cell_id', 'tissue_id'],
    right_on=['cell_id', 'tissue'],
    how='inner',
    validate='one_to_one',
    sort=False,
)
assert len(df) == len(obs) == len(vals) == 36425, 'Cell alignment mismatch'
assert (df['tissue_id'].to_numpy() == df['tissue'].to_numpy()).all()

# Tissue is selected in the original benchmark using *truth variation* only,
# not the relative accuracy of the two prediction methods.
if 'map_tissue' in provenance:
    tissue_id = int(provenance['map_tissue'])
else:
    spreads = (
        df.loc[df['tissue'] > 0].groupby('tissue')['rho_true']
        .apply(lambda v: np.quantile(v, 0.90) - np.quantile(v, 0.10))
    )
    tissue_id = int(spreads.idxmax())
assert 1 <= tissue_id <= 9
label_number = tissue_id + 1  # display convention: Tissue 1 ... Tissue 10

sub = df.loc[df['tissue'].eq(tissue_id)]
x = sub['x'].to_numpy(float)
y = -sub['y'].to_numpy(float)  # publication display flip, not a transformation of data
rho_true = sub['rho_true'].to_numpy(float)
rho_zoetrope = sub['rho_spatial_ti'].to_numpy(float)
stories_minus_j = -sub['stories_native_potential'].to_numpy(float)

assert np.isfinite(np.column_stack([x, y, rho_true,
                                    rho_zoetrope, stories_minus_j])).all()

heatmaps = [
    ('Ground-truth progression', percentile_rank(rho_true)),
    (r'Zoetrope progression $\widehat{\rho}$', percentile_rank(rho_zoetrope)),
    (r'STORIES negative potential $-J$', percentile_rank(stories_minus_j)),
]
print(f'Plotting Tissue {label_number} (frozen ID {tissue_id}), n={len(sub):,} cells')
print('STORIES displayed quantity: negative of frozen native potential (−J).')

# ============================================================
# 3. FIGURE STYLE — MATCH OTHER PUBLICATION PANELS
# ============================================================

fonts_installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((name for name in
             ('Arial', 'Liberation Sans', 'Arimo', 'Nimbus Sans', 'DejaVu Sans')
             if name in fonts_installed), 'DejaVu Sans')
print('Publication font:', FONT)

mpl.rcParams.update({
    'font.family': FONT,
    'font.size': 10.5,
    'axes.titlesize': MAP_TITLE_SIZE,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12.0,
    'axes.linewidth': 0.75,
    'text.color': INK,
    'axes.labelcolor': INK,
    'xtick.color': INK,
    'ytick.color': INK,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'path',  # avoid unsupported fonts in PowerPoint
    'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

# ============================================================
# 4. THREE STACKED TISSUE MAPS
# ============================================================

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

# Tissue rows have a consistent XY extent; the lower row is placed modestly
# closer to the colorbar than in the original version.
positions = [
    (0.08, 0.710, 0.84, 0.188),
    (0.08, 0.447, 0.84, 0.188),
    (0.08, 0.175, 0.84, 0.188),
]

pad_x = max(float(np.ptp(x)), 1e-6) * 0.025
pad_y = max(float(np.ptp(y)), 1e-6) * 0.050
x_lim = (float(x.min()-pad_x), float(x.max()+pad_x))
y_lim = (float(y.min()-pad_y), float(y.max()+pad_y))

for (title, values), pos in zip(heatmaps, positions):
    ax = fig.add_axes(pos)
    sc = ax.scatter(
        x, y,
        c=values,
        cmap='viridis', vmin=0, vmax=1,
        s=2.9, linewidths=0, rasterized=True,
    )
    ax.set_xlim(*x_lim)
    ax.set_ylim(*y_lim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_axis_off()
    ax.set_title(title, fontsize=MAP_TITLE_SIZE, fontweight='bold', pad=4.5)

fig.text(
    0.5, 0.965,
    f'Tissue {label_number}',
    ha='center', va='top', fontsize=PANEL_TITLE_SIZE, fontweight='bold', color=INK,
)

# Larger colorbar, positioned lower to use the available bottom margin.
cax = fig.add_axes([0.17, 0.072, 0.66, 0.026])
cb = fig.colorbar(sc, cax=cax, orientation='horizontal',
                  ticks=[0.0, 0.25, 0.50, 0.75, 1.0])
cb.set_ticklabels(['0', '0.25', '0.5', '0.75', '1'])
cb.ax.tick_params(
    labelsize=COLORBAR_TICK_SIZE, length=2.4, width=0.7, pad=1.4,
)
cb.outline.set_linewidth(0.6)

# Put the label below the tick numbers (no overlap).
label = fig.text(
    0.50, 0.018, 'Within-tissue value percentile',
    ha='center', va='center', fontsize=COLORBAR_LABEL_SIZE,
    fontweight='normal', color=INK,
)

# ============================================================
# 5. CHECK ALL ELEMENTS FIT THE FIXED SVG/PDF CANVAS
# ============================================================

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
bounds = np.asarray(fig.get_tightbbox(renderer).extents, float)
print('Canvas:', FIG_W, 'x', FIG_H, 'in')
print('Rendered content bounds:', [round(float(v), 4) for v in bounds])
TOL = 0.008
assert bounds[0] >= -TOL and bounds[1] >= -TOL, (
    'Plot elements beyond left/bottom canvas edges', bounds
)
assert bounds[2] <= FIG_W + TOL and bounds[3] <= FIG_H + TOL, (
    'Plot elements beyond right/top canvas edges', bounds
)

# Check that the colorbar numbers remain above the bottom label.
label_bbox = label.get_window_extent(renderer)
tick_boxes = [tick.get_window_extent(renderer)
              for tick in cb.ax.get_xticklabels() if tick.get_visible()]
if tick_boxes:
    assert label_bbox.y1 < min(box.y0 for box in tick_boxes), (
        'Colorbar axis label overlaps the tick numbers; increase spacing slightly.'
    )

# ============================================================
# 6. SAVE SVG / PDF / PNG AND PROVENANCE
# ============================================================

OUT.mkdir(parents=True, exist_ok=True)
stem = f'FIG_synthetic_progression_vertical_tissue_{label_number:02d}_negative_STORIES_publication'
for ext in ('svg', 'pdf', 'png'):
    path = OUT / f'{stem}.{ext}'
    fig.savefig(path, dpi=DPI, bbox_inches=None, facecolor='white')
    print('SAVED:', path)

root = ET.parse(OUT / f'{stem}.svg').getroot()
print('SVG dimensions:', root.attrib.get('width'), '×', root.attrib.get('height'))

receipt = {
    'dataset_id': manifest['dataset_id'],
    'dataset_seed': 20261023,
    'tissue_id_zero_based': tissue_id,
    'display_tissue': label_number,
    'tissue_selection': provenance.get('tissue_selection', 'truth progression spread'),
    'input_progression_file': str(values_path),
    'input_progression_sha256': sha256(values_path),
    'input_observed_sha256': sha256(observed_path),
    'stories_orientation': 'negative native potential -J, fixed by design',
    'map_normalization': 'within-tissue percentile rank, separately per quantity',
    'panel_inches': [FIG_W, FIG_H],
    'publication_font': FONT,
    'colorbar_tick_font_points': COLORBAR_TICK_SIZE,
    'colorbar_label_font_points': COLORBAR_LABEL_SIZE,
}
(OUT / f'{stem}_receipt.json').write_text(json.dumps(receipt, indent=2)+'\n')

plt.show()
plt.close(fig)
print('Finished:', OUT)